In [ ]:
import MDAnalysis as mda
import numpy as np
import matplotlib.pyplot as plt
from numba import jit
from tqdm import tqdm
import time

u = mda.Universe("start_drudes.pdb", "FV_NVT.dcd")
u.guess_TopologyAttrs(context='default', to_guess=['elements'])

frameStart = int(len(u.trajectory) / 2) # (抓總幀數的一半開始分析)
#frameStart = 0
frameNeed = len(u.trajectory) - frameStart

#輸出結果來確認:
print(len(u.trajectory))
print(frameStart)
print(frameNeed)

avgfreq=1 # (自己設定)
frameAverage = int(frameNeed / avgfreq)

#輸出結果來確認:
print(frameAverage)

bins = 1000 # (自己設定)
dz = u.trajectory[0].dimensions[2] / bins
cation = u.select_atoms("resname BMI")
anion  = u.select_atoms("resname trf")

#輸出結果來確認:
print(f"Box Z dimension: {u.trajectory[0].dimensions[2]:.2f} Å")
print(f"Bin width (dz): {dz:.4f} Å")
print(f"Number of cations: {len(cation)}")
print(f"Number of anions: {len(anion)}")
print()

# print("陽離子的位置:\n", cation.positions)
# print("陰離子的位置:\n", anion.positions)

# 使用 numba 優化的計算函數
@jit(nopython=True)
def calculate_density_counts(positions_z, dz, bins):
    """使用 numba 加速的密度計算函數"""
    counts = np.zeros(bins, dtype=np.int64)
    for z in positions_z:
        bin_idx = int(z // dz)
        if 0 <= bin_idx < bins:  # 確保索引在範圍內
            counts[bin_idx] += 1

    print("positions_z:", positions_z)
    return counts

# 初始化為 numpy 陣列
cation_counts = np.zeros(bins, dtype=np.int64)
anion_counts = np.zeros(bins, dtype=np.int64)

# 開始計時
start_time = time.time()

for i in tqdm(range(frameAverage), desc="計算離子密度", unit="frame"):
    frameCurrent = frameStart + i * avgfreq
    if frameCurrent >= len(u.trajectory):
        break
        
    u.trajectory[frameCurrent]
    
    # 使用優化後的函數計算
    cation_counts += calculate_density_counts(cation.positions[:, 2], dz, bins)
    anion_counts += calculate_density_counts(anion.positions[:, 2], dz, bins)

# 計算執行時間
elapsed_time = time.time() - start_time
print(f"\n計算完成！耗時: {elapsed_time:.2f} 秒")
print(f"平均每幀處理時間: {elapsed_time/frameAverage*1000:.2f} 毫秒")
print()

# 歸一化 (簡化版，一次完成)
cation_Density = cation_counts / len(cation) / frameAverage / bins
anion_Density = anion_counts / len(anion) / frameAverage / bins

print(f"Cation density sum: {cation_Density.sum():.6f}")
print(f"Anion density sum: {anion_Density.sum():.6f}")
print()

/home/atuo/miniforge3/envs/cuda/lib/python3.13/site-packages/MDAnalysis/topology/PDBParser.py:346: UserWarning: Unknown element  found for some atoms. These have been given an empty element record. If needed they can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn(wmsg)
/home/atuo/miniforge3/envs/cuda/lib/python3.13/site-packages/MDAnalysis/coordinates/DCD.py:165: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


2569
1284
1285
1285
Box Z dimension: 260.42 Å
Bin width (dz): 0.2604 Å
Number of cations: 14000
Number of anions: 6400



計算離子密度:   3%|▎         | 34/1285 [00:00<00:03, 339.03frame/s]

positions_z: [57.77473  57.194412 57.95333  ... 24.293127 24.007944 25.109333]
z: 25.109333038330078
positions_z: [33.23599  32.439583 32.94283  ... 41.764748 42.210094 42.710926]
z: 42.7109260559082
positions_z: [57.253326 56.90736  57.663136 ... 24.74689  24.443014 25.17949 ]
z: 25.179489135742188
positions_z: [34.77139  34.771843 34.39459  ... 42.222656 42.09017  42.269226]
z: 42.26922607421875
positions_z: [57.913082 57.18709  58.121048 ... 24.664627 24.47882  25.806688]
z: 25.80668830871582
positions_z: [33.76756  35.04437  33.058857 ... 39.92493  38.86692  37.73186 ]
z: 37.73186111450195
positions_z: [58.127598 57.442455 58.3296   ... 25.274324 25.24733  25.199047]
z: 25.199047088623047
positions_z: [33.92796  35.16152  33.31211  ... 40.71804  40.419075 38.60883 ]
z: 38.608829498291016
positions_z: [58.37318  57.226852 58.23794  ... 24.439856 24.34184  25.814667]
z: 25.814666748046875
positions_z: [33.447975 34.341278 32.769726 ... 38.735855 41.04528  39.66423 ]
z: 39.66423034667

計算離子密度:  12%|█▏        | 157/1285 [00:00<00:01, 858.87frame/s]

...

positions_z: [59.16504  59.976307 60.158497 ... 23.788359 23.540709 25.039415]
z: 25.03941535949707
positions_z: [34.95117  34.64699  36.073147 ... 41.16371  39.55455  40.954735]
z: 40.954734802246094
positions_z: [59.53146  60.23267  60.50537  ... 24.759487 24.672901 25.596146]
z: 25.596145629882812
positions_z: [34.529156 34.84573  35.335537 ... 41.676388 39.69108  40.51991 ]
z: 40.5199089050293
positions_z: [59.013443 59.977856 60.146664 ... 24.695097 24.521734 25.768604]
z: 25.768604278564453
positions_z: [35.004837 35.850826 35.24036  ... 41.848373 39.67928  40.96632 ]
z: 40.9663200378418
positions_z: [59.174015 60.853626 60.224964 ... 23.957668 24.205292 25.358377]
z: 25.35837745666504
positions_z: [34.416985 35.526806 34.811447 ... 42.30691  39.92849  41.49069 ]
z: 41.49068832397461
positions_z: [59.556515 61.240147 60.728886 ... 23.808922 23.657911 24.757042]
z: 24.757041931152344
positions_z: [35.037823 35.077156 34.10226  ... 41.511414 39.385353 39.924065]
z: 39.9240646362304

計算離子密度:  87%|████████▋ | 1118/1285 [00:01<00:00, 1197.14frame/s]

positions_z: [34.61725  35.467724 35.18318  ... 41.42267  41.479176 41.87654 ]
z: 41.87654113769531
positions_z: [61.850056 62.017696 61.80453  ... 24.013872 23.988    25.418003]
z: 25.41800308227539
positions_z: [34.02582  34.311596 34.626812 ... 41.211483 41.881382 42.37394 ]
z: 42.373939514160156
positions_z: [62.354527 62.08687  62.252552 ... 24.50474  24.305895 25.355013]
z: 25.355012893676758
positions_z: [33.459644 34.300655 33.37003  ... 41.296677 40.420467 42.648396]
z: 42.64839553833008
positions_z: [61.88475  62.030033 61.855934 ... 23.932734 23.60777  24.84754 ]
z: 24.8475399017334
positions_z: [33.91531  34.612747 34.336723 ... 39.614536 41.76976  39.895603]
z: 39.89560317993164
positions_z: [62.7477   62.635895 62.624573 ... 24.311415 24.350454 25.324793]
z: 25.324792861938477
positions_z: [35.143494 36.00909  35.15707  ... 40.897686 42.261066 43.125137]
z: 43.12513732910156
positions_z: [61.92544  62.590527 62.0867   ... 24.248747 24.375618 25.687538]
z: 25.6875381469726

計算離子密度: 100%|██████████| 1285/1285 [00:01<00:00, 1119.56frame/s]

positions_z: [62.40376  63.319523 62.355003 ... 24.444674 24.387548 25.878101]
z: 25.878101348876953
positions_z: [34.956604 35.78676  35.374134 ... 41.015602 40.914577 40.53769 ]
z: 40.537689208984375
positions_z: [62.832783 63.552723 62.52935  ... 24.122087 24.086517 25.251158]
z: 25.251157760620117
positions_z: [33.106525 32.89052  31.985455 ... 40.657486 41.600803 40.934933]
z: 40.934932708740234
positions_z: [62.274677 63.63382  62.766365 ... 24.281954 25.311554 25.81789 ]
z: 25.817890167236328
positions_z: [34.043385 34.779472 34.88956  ... 40.903587 40.60812  40.472366]
z: 40.47236633300781
positions_z: [62.09241  63.53426  62.586037 ... 24.926123 24.56176  25.9508  ]
z: 25.9507999420166
positions_z: [34.377327 34.22768  35.69592  ... 41.033733 40.655537 40.47805 ]
z: 40.478050231933594
positions_z: [61.711803 62.73151  61.670387 ... 24.33664  24.355474 25.730856]
z: 25.73085594177246
positions_z: [33.815968 34.091915 34.932693 ... 41.086506 41.73809  40.745346]
z: 40.7453460693